# Iterators and Iterables — Pythonic Thinking

Loops look simple on the surface: `for i in num:` and Python hands you one item at a time. Underneath that single line sits a tiny, elegant contract between two kinds of objects — the **iterable**, which owns the data, and the **iterator**, which knows how to walk through it. This notebook is about that contract. Once you can see it, a surprising amount of Python (loops, generators, file reading, `zip`, `enumerate`, streaming APIs) stops being magic and becomes the same five lines of protocol repeated everywhere.

Why should you care? Because understanding iterators changes how you judge memory. A list of a hundred million integers can crash your program, while `range(100000000)` sits in memory comfortably — not because Python is clever about lists, but because `range` is an *iterator-style* object that computes each number only when you ask for it. The same idea powers generators, data pipelines, and lazy loading of files or images.

Here is the roadmap for this lesson:

1. Define **iteration**, **iterables** and **iterators**, and see that `iter()` and `next()` are the only two functions you really need.
2. Learn the quick `dir()` trick for telling an iterable from an iterator by looking for `__iter__` and `__next__`.
3. Unpack what a `for` loop actually does — spoiler: it calls `iter()` once, then keeps calling `next()` until `StopIteration` is raised.
4. Rebuild the loop ourselves as `mera_khudka_for_loop`, using `try`/`except StopIteration`.
5. Build our own `range()` from two classes, `mera_range` (the iterable) and `mera_range_iterator` (the iterator), so the protocol stops being abstract.

Keep three definitions in your back pocket. An **iterable** is any object whose `__iter__` method returns an iterator — lists, strings, dicts, sets, `range` objects and files all qualify. An **iterator** is an object with both `__iter__` (returning itself) and `__next__` (returning the next value or raising `StopIteration`). **Iteration** is simply the act of consuming an iterator until it is exhausted. Get those straight and this whole notebook is just variations on them.

# 1. Iteration, Iterables and Iterators

## 1.1 The two-object dance behind every loop

**Iteration** is the process of visiting items one after another — anything a `for` loop does is iteration. The code right below this cell is the plainest version: a list `num = [1, 2, 3]` and a `for` loop printing `1`, `2`, `3`.

An **iterable** is an object that *can* be iterated over: lists, tuples, strings, dicts, sets, `range` objects and files all qualify. Formally, an iterable knows how to produce an iterator when you call `iter()` on it — think of a cinema as the iterable: it can issue many tickets, show after show.

An **iterator** is the ticket holder: one object that walks a sequence step by step. It has exactly two powers — `iter()`, which simply returns itself, and `next()`, which returns the following value. When there is nothing left, `next()` raises `StopIteration`, the official end-of-the-road signal.

Two consequences matter in practice:

- **Iterators are lazy.** They never build the whole sequence in advance. The second cell confirms it: a list comprehension of almost ten thousand integers costs real memory, while `range(1, 10000000000)` costs almost nothing, because a `range` stores only a start, a stop and a step.
- **Iterators are one-shot.** Once exhausted, they are done — you must call `iter()` again to get a fresh walk.

Tiny taste: `it = iter([1, 2, 3])` creates a `list_iterator`; `next(it)` gives `1`, `next(it)` gives `2`, and one more call after `3` raises `StopIteration`. Those two functions are the entire protocol — the cells below simply prove it on real objects.

In [1]:
# Example
num = [1,2,3]

for i in num:
    print(i)

1
2
3


## What is Iterator

An Iterator is an object that allows the programmer to traverse through a sequence of data without having to store the entire data in the memory

In [2]:
# Example
L = [x for x in range(1,10000)]

#for i in L:
    #print(i*2)
    
import sys

print(sys.getsizeof(L)/64)

x = range(1,10000000000)

#for i in x:
    #print(i*2)
    
print(sys.getsizeof(x)/64)



1330.875
0.75


## What is Iterable
Iterable is an object, which one can iterate over

 It generates an Iterator when passed to iter() method.

### Semantics
Iterables can produce iterators; iterators implement __iter__() and __next__(), support lazy evaluation and one-pass iteration.

### Why
Memory efficient for large streams; enables Python iteration protocol (for loops).

### When to use
Large/infinite sequences, streaming data.

### Common mistakes
Reusing exhausted iterators without recreating.

In [3]:
# Example

L = [1,2,3]
type(L)


# L is an iterable
type(iter(L))

# iter(L) --> iterator

list_iterator

## Point to remember — the two rules

- **Every Iterator is also an Iterable.** An iterator implements `__iter__` and returns itself, so `iter(some_iterator)` is always allowed and gives back the very same object.
- **Not every Iterable is an Iterator.** A list, tuple, dict, set or string can be looped over, but it keeps no cursor of its own. Calling `next([1, 2, 3])` raises `TypeError`, because a list has no `__next__`.

| | Iterable | Iterator |
| --- | --- | --- |
| Must define | `__iter__()` | `__iter__()` **and** `__next__()` |
| Job | Produce iterators (a factory) | Track position and yield values one by one |
| Examples | `[1, 2, 3]`, `"hello"`, `range(10)`, `{1:2}` | `iter([1, 2, 3])`, an open file, a generator |

Analogy time. A playlist is iterable — it can spawn players. Each player object is an iterator: it remembers the current track and moves forward. You can start several players from the same playlist, and each keeps its own position. Calling `iter()` on a player just resumes that same player rather than starting a new one — exactly what the `id()` experiment further down demonstrates.

## Trick — recognise them with `dir()`

You never have to guess whether an object is iterable or an iterator — ask Python for its methods.

- `dir(obj)` without `__iter__` → the object is neither iterable nor an iterator. An `int` is like this: `for i in a:` on `a = 2` would raise `TypeError: 'int' object is not iterable`.
- `dir(obj)` with `__iter__` but without `__next__` → iterable, not an iterator. Lists, dicts and strings land here.
- `dir(obj)` with both `__iter__` and `__next__` → a full-fledged iterator. Generators and `list_iterator` objects look like this.

So the trick is a two-step checklist: look for `__iter__` first (the door), then look for `__next__` (the engine). The `dir(a)` and `dir(T)` calls in the next cells exist precisely for this comparison — the integer `a` shows no iteration methods at all, while the dict displays `__iter__` proudly (and `__getitem__`, which also helps loops). Everything else in those long dumps — `__add__`, `__eq__`, `__hash__` and friends — belongs to the object's other duties and can be ignored for now.

Mnemonic: **iterable = has a door (`__iter__`); iterator = has a door and an engine (`__iter__` + `__next__`).**

In [4]:
a = 2
a

#for i in a:
    #print(i)
    
dir(a)

['__abs__',
 '__add__',
 '__and__',
 '__bool__',
 '__ceil__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__divmod__',
 '__doc__',
 '__eq__',
 '__float__',
 '__floor__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getnewargs__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__index__',
 '__init__',
 '__init_subclass__',
 '__int__',
 '__invert__',
 '__le__',
 '__lshift__',
 '__lt__',
 '__mod__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__or__',
 '__pos__',
 '__pow__',
 '__radd__',
 '__rand__',
 '__rdivmod__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rfloordiv__',
 '__rlshift__',
 '__rmod__',
 '__rmul__',
 '__ror__',
 '__round__',
 '__rpow__',
 '__rrshift__',
 '__rshift__',
 '__rsub__',
 '__rtruediv__',
 '__rxor__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 '__truediv__',
 '__trunc__',
 '__xor__',
 'as_integer_ratio',
 'bit_count',
 'bit_length',
 'conjugate',
 'denominator',
 'from_bytes',
 'imag',
 'is_integer',
 

In [5]:
T = {1:2,3:4}
dir(T)

['__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__delitem__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__ior__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__ne__',
 '__new__',
 '__or__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed__',
 '__ror__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'clear',
 'copy',
 'fromkeys',
 'get',
 'items',
 'keys',
 'pop',
 'popitem',
 'setdefault',
 'update',
 'values']

In [6]:
L = [1,2,3]

# L is not an iterator
iter_L = iter(L)

# iter_L is an iterator

# 2. How a `for` loop really works

## 2.1 Desugaring `for` into `iter()` + `next()`

Python's `for` statement is pure convenience. Behind the scenes every `for i in num:` is rewritten into a fixed four-step dance:

1. Ask the iterable for an iterator: `it = iter(num)`.
2. Pull a value: `value = next(it)`.
3. Bind the loop variable and run the loop body.
4. Repeat from step 2 — and the moment `next(it)` raises `StopIteration`, Python **silently** ends the loop and continues with whatever comes after it.

That is the whole feature. There is no hidden list, no index counter, no `range(len(...))` anywhere. This is why `for` works identically on lists, strings, dicts, sets, `range` objects, generators and your own custom classes: they all honour the same `__iter__`/`__next__` contract.

The first code cell after this header shows the friendly syntax. The second shows the raw version: `iter_num = iter(num)` fetches the iterator, then each `next()` call pulls one element — `1`, then `2`, then `3`. One extra `next()` call is included on purpose: the list is already exhausted, so it raises `StopIteration`, the very exception that normally ends a loop invisibly.

Two practical takeaways. First, mutating a collection while a `for` loop walks it is dangerous, because you are disturbing the walker's position in the middle of its journey. Second, any object that implements `__iter__` + `__next__` becomes instantly loopable — no inheritance from a base class required, just the protocol.

In [7]:
num = [1,2,3]

for i in num:
    print(i)

1
2
3


In [8]:
num = [1,2,3]

# fetch the iterator
iter_num = iter(num)

# step2 --> next
next(iter_num)
next(iter_num)
next(iter_num)
next(iter_num)

StopIteration: 

# 3. Writing our own `for` loop

## 3.1 `while True` plus `try/except StopIteration`

If a `for` loop is just `iter()` plus repeated `next()` until `StopIteration`, we can rebuild it in about eight lines. That is exactly what `mera_khudka_for_loop(iterable)` does:

```python
iterator = iter(iterable)      # step 1: get a walker
while True:
    try:
        print(next(iterator))  # step 2: pull one value
    except StopIteration:      # step 3: the walker is out of values
        break
```

Read the shape carefully. `iter(iterable)` is called **first**, on whatever the caller passed — a list, a `range`, a tuple, a set or a dict — so the function accepts any iterable, not just lists. The `while True` loop never tests a length or an index; the only exit door is the `StopIteration` exception. In Python, exceptions are not just error reports — they are normal control flow, and `StopIteration` is the official "no more items" message.

The demo cell then runs the custom loop over five containers, and for the dict it prints the keys `0` and `1`, because dicts iterate over keys by default — the same rule real `for` loops follow.

Notice what our version *cannot* do yet: it prints values instead of handing them back, so it cannot feed another loop as a data source. Fixing that is the job of generators, covered in the next notebook. For now, the point is proven: a `for` loop has no magic — just protocol plus exception handling.

In [9]:
def mera_khudka_for_loop(iterable):
    
    iterator = iter(iterable)
    
    while True:
        
        try:
            print(next(iterator))
        except StopIteration:
            break           

In [10]:
a = [1,2,3]
b = range(1,11)
c = (1,2,3)
d = {1,2,3}
e = {0:1,1:1}

mera_khudka_for_loop(e)

0
1


# 4. `iter()` on an iterator returns itself

## 4.1 One object, two names

Here is a detail that confuses almost everyone the first time: calling `iter()` on something that is *already* an iterator does not create a second iterator — it returns the **same object**.

The experiment makes this visible. `iter_obj = iter([1, 2, 3])` creates a fresh `list_iterator`, and its `id()` is printed. Then `iter_obj2 = iter(iter_obj)` calls `iter()` on that iterator — and the printed `id()` is identical. Two names, one object.

Why must it work this way? Because `for` loops (and our custom loop from the previous section) *always* call `iter()` on their target before calling `next()`. If `iter()` on an iterator produced a brand-new iterator, every loop would silently restart from the beginning. The rule keeps things sane:

- `iter(iterable)` → a **new** iterator each time (a fresh walk over the same data),
- `iter(iterator)` → the **same** iterator; the walk continues where it stopped.

This also explains why iterators are legal iterables: they pass the first check (`__iter__` exists) while preserving their single cursor. So the relationship is subtle but clean — every iterator is an iterable, but an iterator's iterable-ness is a pass-through, never a rewind.

In [11]:
num = [1,2,3]
iter_obj = iter(num)

print(id(iter_obj),'Address of iterator 1')

iter_obj2 = iter(iter_obj)
print(id(iter_obj2),'Address of iterator 2')

2369496748704 Address of iterator 1
2369496748704 Address of iterator 2


# 5. Building our own `range()`

## 5.1 One iterable, one iterator

`range()` is the perfect teaching tool because its behaviour is familiar: `range(1, 11)` should produce `1, 2, ..., 10` every time you loop over it. Recreating it forces us to split the two roles we have been discussing into two separate classes.

**`mera_range` (the iterable).** Its only job is to hold the data — `start` and `end` — and to hand out iterators. Its `__iter__` method returns a brand-new `mera_range_iterator(self)`, so the same `mera_range` object can spawn fresh walkers, like a playlist spawning new players.

**`mera_range_iterator` (the iterator).** It keeps a reference back to the iterable and does the actual walking:

- `__iter__` returns `self` — the pass-through rule from the previous section,
- `__next__` checks `if self.iterable.start >= self.iterable.end:` and raises `StopIteration` when the range is finished,
- otherwise it returns the current number and advances the cursor.

The verification cells then show `type(x)` is `__main__.mera_range`, and `iter(x)` hands back a `mera_range_iterator` — iterable and iterator, cleanly separated.

One honest caveat about *this* particular implementation: the iterator advances the cursor by mutating `self.iterable.start` on the shared `mera_range` object, so numbers are consumed in place. After one complete pass, `x.start` already equals `x.end`, and a second loop over the same `x` would produce nothing. Built-in `range` avoids this by keeping its start and stop immutable and giving each iterator its own private copy of the position — a good exercise to attempt after this notebook.

In [12]:
class mera_range:
    
    def __init__(self,start,end):
        self.start = start
        self.end = end
        
    def __iter__(self):
        return mera_range_iterator(self)

In [13]:
class mera_range_iterator:
    
    def __init__(self,iterable_obj):
        self.iterable = iterable_obj
    
    def __iter__(self):
        return self
    
    def __next__(self):
        
        if self.iterable.start >= self.iterable.end:
            raise StopIteration
            
        current = self.iterable.start
        self.iterable.start+=1
        return current

In [14]:
x = mera_range(1,11)

In [15]:
type(x)

__main__.mera_range

In [16]:
iter(x)

## Summary

An **iterable** owns data and knows how to produce iterators via `__iter__`; an **iterator** carries the cursor and provides `__iter__` (returning itself) and `__next__` (returning one value or raising `StopIteration`). Every iterator is iterable, but not the other way around — you can tell them apart with `dir()` and the presence of `__iter__`/`__next__`. A `for` loop is nothing more than one `iter()` call followed by repeated `next()` calls until `StopIteration` stops it silently; we proved it by rebuilding the loop ourselves with `while True` and `try/except`. Calling `iter()` on an iterator returns the same object, which is why loops never accidentally rewind. Finally, we designed our own `range()` as a pair of classes — `mera_range` producing iterators and `mera_range_iterator` doing the stepping — the exact blueprint that generators automate in the next notebook.